# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR² dataset on extracellular vesicles from stimulated human mast cells using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema available at:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and explore basic information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get and display dataset metadata
metadata_obj = dataset.metadata
print(f"Dataset name: {metadata_obj.name}")
print(f"Description: {metadata_obj.description}")

## 2. Data Overview
List available record sets, their `@id` values, and contained fields. All entities are referenced by their `@id`.

In [ ]:
# Discover all record sets and their fields by @id
record_sets = []
try:
    for rset in dataset.record_sets:
        print(f"Record Set: {rset.name}")
        print(f"  @id: {rset['@id']}")
        if hasattr(rset, 'fields'):
            print("  Fields:")
            for field in rset.fields:
                print(f"    @id: {field['@id']}, name: {field.name}")
        print()
        record_sets.append(rset['@id'])
except Exception as e:
    print("Could not list record sets and fields.")

if len(record_sets):
    print(f"Found Record Set @id values: {record_sets}")
else:
    print("No record sets found. Check the dataset schema.")

## 3. Data Extraction
Load a record set by its `@id` into a DataFrame for analysis. Use the `@id`s from the previous overview.

> **Note:** If no record sets appear above, check the Croissant schema for available tables or fields. For this dataset, the primary table is typically named or described as 'protein_table', 'main', or similar. Let's try to enumerate them via the API.

In [ ]:
# Try automatically extracting all record sets if possible
# (If none found, note and move to custom extraction)
if len(record_sets) > 0:
    dataframes = {}
    for record_set_id in record_sets:
        print(f"Attempting to load data for record set: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} rows. Columns: {df.columns.tolist()}")
        except Exception as e:
            print(f"Failed to load records for {record_set_id}: {e}")
else:
    print("No record sets to load.")

### If no record sets are publicly visible via the schema, try to find distributions/files:
For datasets distributed as a single tabular file, you can try to access them directly via the `dataset.distributions` attribute.

In [ ]:
# Try loading from the main distribution if no classic record sets are present
if not locals().get('dataframes') or not dataframes:
    table_found = False
    for dist in getattr(dataset, 'distributions', []):
        if hasattr(dist, 'media_type') and 'csv' in dist.media_type.lower():
            print(f"Found CSV distribution: {dist['@id']}")
            # Try loading as a record set for this CSV
            records = list(dataset.records(distribution=dist['@id']))
            df = pd.DataFrame(records)
            dataframes = {dist['@id']: df}
            table_found = True
            print(f"Loaded {len(df)} rows. Columns: {df.columns.tolist()}")
            break
    if not table_found:
        print("No CSV or table-like distribution found in dataset.")
# If both approaches fail, please examine metadata.info() output.

Inspect the columns of the loaded DataFrame. Reference field names by their `@id` as available.

In [ ]:
# Display columns and first few records
if locals().get('dataframes'):
    first_key = list(dataframes.keys())[0]
    print(f"Columns for record set or distribution: {first_key}")
    print(dataframes[first_key].columns.tolist())
    display(dataframes[first_key].head())
else:
    print("No dataframes were loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic analysis:
- Filtering records based on a numeric field (e.g., `cr:peptide_count`).
- Normalizing a numeric column.
- Grouping by a field, if available.

> Be sure to use the correct `@id` of fields as column names if available in your DataFrame.

In [ ]:
df = None
df_key = None
if locals().get('dataframes'):
    df_key = list(dataframes.keys())[0]
    df = dataframes[df_key]

# Select a likely numeric field and group field by name/ID
# Candidates for this dataset: 'cr:peptide_count', 'cr:coverage', 'cr:MW', 'cr:pI', 'cr:sample_A', etc.
numeric_field = None
group_field = None

candidates = ['cr:peptide_count', 'peptide_count', 'cr:MW', 'MW', 'coverage', 'cr:coverage', 'cr:pI', 'pI', 'normalized_abundance', 'sample_A', 'sample_B', 'sample_C']
if df is not None:
    for candidate in candidates:
        if candidate in df.columns:
            numeric_field = candidate
            break
    # Try grouping by a non-numeric field, prefer accession/description
    group_candidates = ['cr:accession', 'accession', 'cr:description', 'description']
    for candidate in group_candidates:
        if candidate in df.columns:
            group_field = candidate
            break

if df is not None and numeric_field is not None:
    # Clean numeric
    df = df[df[numeric_field].apply(lambda x: pd.notnull(x) and str(x).replace('.', '', 1).replace('-', '', 1).isnumeric())]
    df[numeric_field] = df[numeric_field].astype(float)
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} ({filtered_df.shape[0]} rows)")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} (first few rows):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if found
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        print(f"\nMean {numeric_field} grouped by {group_field} (top rows):")
        print(grouped_df.head())
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization
Visualize the distribution and comparison of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field], kde=True, bins=30, color='dodgerblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(12, 5))
        top_groups = df[group_field].value_counts().index[:10]
        sns.boxplot(data=df[df[group_field].isin(top_groups)], x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field} (top 10 groups)")
        plt.xticks(rotation=60)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and perform initial exploration on the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using the Croissant data model and `mlcroissant`. We extracted data using standardized `@id` references, filtered and normalized peptide or protein abundance fields, and visualized their distributions.

#### Key Insights:
- The dataset's main record set contains protein/peptide quantification and annotation fields.
- Filtering and normalization surface potential lead candidates for downstream analysis such as clustering or differential analysis.
- All table fields and sets should be referenced by their Croissant `@id` for robust, schema-aligned extraction.

Further work might leverage the full field dictionary, advanced groupings, or explore integration with biological ontologies.

For more, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/latest/).